## EDA Insights: Customer Churn Patterns

### 1. Overall Churn Rate

 churn | count |  pct
-------+-------+-------
 f     |  5174 | 73.46
 t     |  1869 | 26.54

- Tổng số khách hàng: **7,043**
- Tỷ lệ churn: 26.54% [1,869] khách hàng đã rời bỏ dịch vụ
- Đây là bài toán **imbalanced classification** ở mức trung bình (~1:3) — sẽ cần dùng
  `class_weight='balanced'` hoặc điều chỉnh threshold khi train model, vì
  accuracy làm metric chính (accuracy sẽ bị đánh lừa bởi nhóm đa số).

---

### 2. Churn theo loại hợp đồng (Contract Type)
    contract    | total | churn_rate_pct
----------------+-------+----------------
 Month-to-month |  3875 |          42.71
 One year       |  1473 |          11.27
 Two year       |  1695 |           2.83

**Insight:** Khách hàng hợp đồng Month-to-month có churn rate cao gấp **~15 lần**
so với Two year. Điều này gợi ý `contract` là một trong những feature dự đoán mạnh
nhất — về mặt business, có thể do khách hàng ngắn hạn chưa có cam kết dài hạn,
dễ chuyển sang nhà cung cấp khác. **Gợi ý hành động:** công ty có thể ưu đãi để
khuyến khích khách hàng Month-to-month chuyển sang hợp đồng dài hạn hơn.

---

### 3. Churn theo thời gian sử dụng dịch vụ (Tenure)

 tenure_group | churn_rate_pct | total
--------------+----------------+-------
 0-12 months  |          47.44 |  2186
 13-24 months |          28.71 |  1024
 25-48 months |          20.39 |  1594
 49+ months   |           9.51 |  2239

**Insight:** Churn rate giảm dần rõ rệt theo thời gian gắn bó — khách hàng mới
(dưới 1 năm) có nguy cơ rời bỏ cao nhất. Điều này cho thấy **giai đoạn onboarding**
là thời điểm quan trọng nhất để giữ chân khách hàng. `tenure` nên được đưa vào
model như 1 feature liên tục, không cần rời rạc hóa.

---

### 4. Churn theo phương thức thanh toán (Payment Method)
      payment_method       | total | churn_rate_pct
---------------------------+-------+----------------
 Electronic check          |  2365 |          45.29
 Mailed check              |  1612 |          19.11
 Bank transfer (automatic) |  1544 |          16.71
 Bank transfer (automatic) |  1544 |          16.71
 Credit card (automatic)   |  1522 |          15.24
 Bank transfer (automatic) |  1544 |          16.71
 Credit card (automatic)   |  1522 |          15.24
 Bank transfer (automatic) |  1544 |          16.71
 Credit card (automatic)   |  1522 |          15.24
 Bank transfer (automatic) |  1544 |          16.71
 Credit card (automatic)   |  1522 |          15.24



**Insight:** Khách hàng dùng "Electronic check" có churn rate cao gấp ~3 lần so
với các phương thức thanh toán tự động (bank transfer, credit card). Giả thuyết:
thanh toán thủ công đòi hỏi hành động chủ động mỗi tháng, tạo điểm "ma sát"
khiến khách hàng dễ cân nhắc huỷ dịch vụ hơn so với thanh toán tự động.

---

### 5. Churn theo loại Internet Service
 internet_service | total | churn_rate_pct
------------------+-------+----------------
 Fiber optic      |  3096 |          41.89
 DSL              |  2421 |          18.96
 No               |  1526 |           7.40

**Insight:** Khách hàng dùng Fiber optic có churn rate cao nhất dù đây là dịch vụ
cao cấp hơn DSL. Giả thuyết: giá cao hơn hoặc vấn đề kỹ thuật/chất lượng dịch vụ
khiến nhóm này dễ rời bỏ hơn — cần kiểm tra thêm mối liên hệ với `monthly_charges`
để xác nhận giả thuyết về giá.

---

### 6. So sánh chi phí giữa nhóm Churn và Không Churn
 churn | avg_monthly | avg_total
-------+-------------+-----------
 f     |       61.27 |   2555.34
 t     |       74.44 |   1531.80

**Insight:** Nhóm churn có `monthly_charges` trung bình cao hơn nhưng
`total_charges` lại thấp hơn nhiều — điều này nhất quán với insight ở mục 3
(nhóm churn thường là khách hàng mới, tenure ngắn nên total_charges tích lũy
ít hơn, dù trả phí hàng tháng cao hơn).

---

### Tổng kết: 3 yếu tố ảnh hưởng churn mạnh nhất
1. **Contract type** — Month-to-month có nguy cơ churn cao nhất
2. **Tenure** — khách hàng mới (<12 tháng) rủi ro cao nhất
3. **Payment method** — Electronic check liên quan tới churn cao hơn đáng kể

### Quyết định cho bước Modeling
- Xử lý imbalance bằng `class_weight='balanced'`, không dùng SMOTE (mức lệch
  26.5% chưa đủ nghiêm trọng để cần sinh dữ liệu giả).
- Metric đánh giá chính: **F1-score** và **ROC-AUC** thay vì accuracy.
- Ưu tiên các feature: `contract`, `tenure`, `payment_method`, `internet_service`,
  `monthly_charges` làm input chính cho model.